# Project 97 — Local API Spec Explainer
## OpenAPI Spec → Plain English Docs → SDK Examples

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Sample API Specification

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

api_spec = {
    "openapi": "3.0.0",
    "info": {"title": "Task Manager API", "version": "2.0"},
    "paths": {
        "/tasks": {
            "get": {
                "summary": "List tasks",
                "parameters": [
                    {"name": "status", "in": "query", "schema": {"type": "string", "enum": ["pending","done","archived"]}},
                    {"name": "limit", "in": "query", "schema": {"type": "integer", "default": 20}},
                    {"name": "offset", "in": "query", "schema": {"type": "integer", "default": 0}},
                ],
                "responses": {"200": {"description": "Array of tasks"}, "401": {"description": "Unauthorized"}},
            },
            "post": {
                "summary": "Create task",
                "requestBody": {"content": {"application/json": {"schema": {
                    "type": "object",
                    "required": ["title"],
                    "properties": {
                        "title": {"type": "string", "maxLength": 200},
                        "description": {"type": "string"},
                        "priority": {"type": "string", "enum": ["low","medium","high"]},
                        "due_date": {"type": "string", "format": "date"},
                    }
                }}}},
                "responses": {"201": {"description": "Task created"}, "400": {"description": "Validation error"}},
            },
        },
        "/tasks/{id}": {
            "get": {"summary": "Get task by ID", "responses": {"200": {}, "404": {}}},
            "put": {"summary": "Update task", "responses": {"200": {}, "404": {}}},
            "delete": {"summary": "Delete task", "responses": {"204": {}, "404": {}}},
        },
    },
}
print(f"API: {api_spec['info']['title']} v{api_spec['info']['version']}")
print(f"Endpoints: {sum(len(v) for v in api_spec['paths'].values())}")

## Step 2 — Generate Plain English Documentation

In [ ]:
class EndpointDoc(BaseModel):
    method: str
    path: str
    description: str = Field(description="Plain English explanation")
    use_case: str
    required_params: list[str]
    optional_params: list[str]
    success_response: str
    error_responses: list[str]
    rate_limit_note: str = ""

class APIGuide(BaseModel):
    title: str
    overview: str
    authentication: str
    base_url_example: str
    endpoints: list[EndpointDoc]
    common_workflows: list[str]
    error_handling_tips: list[str]

guide_gen = llm.with_structured_output(APIGuide)

guide = guide_gen.invoke(
    f"Create a beginner-friendly API guide from this OpenAPI spec:\n\n"
    f"{json.dumps(api_spec, indent=2)}"
)

print(f"API Guide: {guide.title}")
print(f"Overview: {guide.overview}")
print(f"\nEndpoints ({len(guide.endpoints)}):")
for ep in guide.endpoints:
    print(f"  {ep.method.upper()} {ep.path}")
    print(f"    {ep.description}")
    print(f"    Use case: {ep.use_case}")

## Step 3 — Generate SDK Code Examples

In [ ]:
sdk_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate code examples for this API endpoint in the specified language. "
     "Include error handling and comments."),
    ("human", "Endpoint: {method} {path}\nDescription: {desc}\nLanguage: {lang}\nCode:")
])
sdk_chain = sdk_prompt | llm | StrOutputParser()

languages = ["Python (requests)", "JavaScript (fetch)", "cURL"]
for ep in guide.endpoints[:3]:
    print(f"\n{'='*50}")
    print(f"{ep.method.upper()} {ep.path}")
    for lang in languages:
        example = sdk_chain.invoke({
            "method": ep.method, "path": ep.path,
            "desc": ep.description, "lang": lang,
        })
        print(f"\n  [{lang}]")
        print(f"  {example[:200]}...")

## Step 4 — Generate Quickstart Guide

In [ ]:
quickstart_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a 'Getting Started in 5 Minutes' guide for this API. "
     "Include: setup, first request, common patterns, error handling."),
    ("human", "API: {title}\nEndpoints: {endpoints}\n\nQuickstart:")
])
quickstart_chain = quickstart_prompt | llm | StrOutputParser()

quickstart = quickstart_chain.invoke({
    "title": guide.title,
    "endpoints": json.dumps([{"method": e.method, "path": e.path, "desc": e.description}
                             for e in guide.endpoints]),
})
print("QUICKSTART GUIDE")
print("=" * 50)
print(quickstart[:600])

## What You Learned
- **OpenAPI → plain English** documentation
- **Multi-language SDK examples** from specs
- **Quickstart guide generation**
- **API documentation automation** pipeline